In [ ]:
import pandas as pd
import os

# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "merged_nscg_2010_to_2021.csv")
df = pd.read_csv(file_path)

In [ ]:
print(len(df))

# Standardize values


In [ ]:
# List of variables to standardize [degree type related]
dg_vars = ['D2DG', 'D3DG', 'D4DG', 'D5DG']

# Before conversion
print("Before conversion:")
for var in dg_vars:
    print(f"\n{var} unique values:", df[var].unique())

# Convert float values to strings without decimal points
for var in dg_vars:
    # Convert non-'L' values to integers (this will handle both '1.0' and '1' to 1)
    mask = (df[var] != 'L') & df[var].notna()
    df.loc[mask, var] = df.loc[mask, var].astype(float).astype(int).astype(str)
    
# After conversion
print("\nAfter conversion:")
for var in dg_vars:
    print(f"\n{var} value counts:")
    print(df[var].value_counts(dropna=False))

In [ ]:
print(df['DGRDG'].unique())

# Filtering

In [ ]:
# 1. valid cohort 'BAYR'
df = df[df['BAYR']< 9998.0]

print(f'# Obs valid cohort:{len(df)}')

# 2. necessary salary range

df = df[df['SALARY'] > 0]

print(f'# Obs positive salary:{len(df)}')


In [ ]:
df['SALARY'].describe()

# merge UR data

In [ ]:
# Define the path and read the CSV files
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_25_34y.csv")
ur_ma_path = os.path.join(base_path, "ur_ma_25_34y.csv")

# Read the files
ur_ba_25_34 = pd.read_csv(ur_ba_path)
ur_ma_25_34 = pd.read_csv(ur_ma_path)
# Rename columns
ur_ba_25_34 = ur_ba_25_34.rename(columns={'CGBD2534': 'UR'})
ur_ma_25_34 = ur_ma_25_34.rename(columns={'CGMD2534': 'UR'})

In [ ]:
ur_ba_25_34

In [ ]:
# First prepare yearly unemployment rates for both degree types
# For Bachelor's (25-34)
ur_ba_25_34['observation_date'] = pd.to_datetime(ur_ba_25_34['observation_date']) 
ur_ba_25_34['year'] = ur_ba_25_34['observation_date'].dt.year
ur_ba_25_34_yearly = ur_ba_25_34.groupby('year')['UR'].mean().reset_index()
ur_ba_25_34_yearly = ur_ba_25_34_yearly.rename(columns={'year': 'BAYR', 'UR': 'gradur_ba'})

# For Master's (25-34) 
ur_ma_25_34['observation_date'] = pd.to_datetime(ur_ma_25_34['observation_date'])
ur_ma_25_34['year'] = ur_ma_25_34['observation_date'].dt.year  
ur_ma_25_34_yearly = ur_ma_25_34.groupby('year')['UR'].mean().reset_index()
ur_ma_25_34_yearly = ur_ma_25_34_yearly.rename(columns={'year': 'BAYR', 'UR': 'gradur_ma'})

# Create separate dataframes for bachelor's and master's degree holders
df_ba = df[df['DGRDG'] == 1].copy()  
df_ma = df[df['DGRDG'] == 2].copy()

# Merge unemployment rates
df_ba = df_ba.merge(ur_ba_25_34_yearly, on='BAYR', how='left') 
df_ma = df_ma.merge(ur_ma_25_34_yearly, on='BAYR', how='left')

# Combine back and create final gradur column  
df_ba['gradur'] = df_ba['gradur_ba']
df_ma['gradur'] = df_ma['gradur_ma'] 
df_final = pd.concat([df_ba, df_ma])

# Check the results
print("Bachelor's degrees:")
print(df_final[df_final['DGRDG']==1][['BAYR', 'gradur']].sort_values('BAYR').drop_duplicates().to_string(index=False))

print("\nMaster's degrees:")  
print(df_final[df_final['DGRDG']==2][['BAYR', 'gradur']].sort_values('BAYR').drop_duplicates().to_string(index=False))

# Check sample sizes
print("\nSample sizes:")
print(df_final.groupby(['BAYR', 'DGRDG'])['gradur'].agg(['count', 'mean']).round(2))

In [ ]:
# Check for missing values before preparing regression variables
print("Missing values in original data:")
print(df_final[['SALARY', 'BAYR', 'REFYR', 'DGRDG', 'gradur']].isnull().sum())

# Clean the data
df_final = df_final.dropna(subset=['SALARY', 'BAYR', 'REFYR', 'DGRDG', 'gradur'])

In [ ]:
len(df_final)

In [ ]:
import numpy as np
import seaborn as sns
import statsmodels.api as sm

# Now prepare regression variables with clean data
# 1. Create log salary
df_final['log_salary'] = np.log(df_final['SALARY'])

# 2. Create experience variables
df_final['exp'] = df_final['REFYR'] - df_final['BAYR']
df_final['exp2'] = df_final['exp']**2

# 3. Create dummy for master's degree
df_final['master'] = (df_final['DGRDG'] == 2).astype(int)

# 4. Create interaction terms
df_final['gradur_master'] = df_final['gradur'] * df_final['master']
df_final['gradur_exp'] = df_final['gradur'] * df_final['exp']
df_final['gradur_master_exp'] = df_final['gradur'] * df_final['master'] * df_final['exp']

# Check for any missing values in regression variables
reg_vars = ['log_salary', 'gradur', 'exp', 'exp2', 'master', 
            'gradur_master', 'gradur_exp', 'gradur_master_exp']
print("\nMissing values in regression variables:")
print(df_final[reg_vars].isnull().sum())

# 5. Create year dummies
year_dummies = pd.get_dummies(df_final['REFYR'], prefix='year')
base_year = min(df_final['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables for regression
X = pd.concat([
    df_final[['gradur', 'gradur_exp', 'master', 'gradur_master', 'gradur_master_exp', 
              'exp', 'exp2']], 
    year_dummies
], axis=1)

# Final check for inf or nan values
print("\nInf values in X:")
print(np.isinf(X).sum())
print("\nNaN values in X:")
print(np.isnan(X).sum())

# Add constant
X = sm.add_constant(X)

# Run regression with clean data
model = sm.OLS(df_final['log_salary'], X)
results = model.fit(cov_type='HC1')

## log(salary)_it = α + β1(gradur)_c + β2(gradur)_c × exp_it + β3master_i + β4(gradur)_c × master_i + β5(gradur)_c × master_i × exp_it + δ1exp_it + δ2exp^2_it + γ_t + ε_it.

In [ ]:
# Use the merged and corrected df_final for regression
# 1. Create log salary
df_final['log_salary'] = np.log(df_final['SALARY'])

# 2. Create experience variables
df_final['exp'] = df_final['REFYR'] - df_final['BAYR']
df_final['exp2'] = df_final['exp']**2

# 3. Create dummy for master's degree
df_final['master'] = (df_final['DGRDG'] == 2).astype(int)

# 4. Create interaction terms
df_final['gradur_master'] = df_final['gradur'] * df_final['master']
df_final['gradur_exp'] = df_final['gradur'] * df_final['exp']
df_final['gradur_master_exp'] = df_final['gradur'] * df_final['master'] * df_final['exp']

# 5. Create year dummies
year_dummies = pd.get_dummies(df_final['REFYR'], prefix='year')
# Drop one year (2010) as base year to avoid multicollinearity
base_year = min(df_final['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables for regression
X = pd.concat([
   df_final[['gradur', 'gradur_exp', 'master', 'gradur_master', 'gradur_master_exp', 
             'exp', 'exp2']], 
   year_dummies
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_final['log_salary'], X)
results = model.fit(cov_type='HC1')  # Using robust standard errors

# Print results
print(results.summary())

# Print detailed interpretation of key coefficients
print("\nDetailed Interpretation of Key Coefficients:")
print("\n1. Initial effect of unemployment rate for bachelor's (β1):")
print(f"Coefficient: {results.params['gradur']:.4f}")
print(f"Standard Error: {results.bse['gradur']:.4f}")
print(f"P-value: {results.pvalues['gradur']:.4f}")

print("\n2. How unemployment effect changes with experience for bachelor's (β2):")
print(f"Coefficient: {results.params['gradur_exp']:.4f}")
print(f"Standard Error: {results.bse['gradur_exp']:.4f}")
print(f"P-value: {results.pvalues['gradur_exp']:.4f}")

print("\n3. Baseline master's premium (β3):")
print(f"Coefficient: {results.params['master']:.4f}")
print(f"Standard Error: {results.bse['master']:.4f}")
print(f"P-value: {results.pvalues['master']:.4f}")

print("\n4. Additional initial effect of unemployment for master's (β4):")
print(f"Coefficient: {results.params['gradur_master']:.4f}")
print(f"Standard Error: {results.bse['gradur_master']:.4f}")
print(f"P-value: {results.pvalues['gradur_master']:.4f}")

print("\n5. How master's unemployment effect changes with experience (β5):")
print(f"Coefficient: {results.params['gradur_master_exp']:.4f}")
print(f"Standard Error: {results.bse['gradur_master_exp']:.4f}")
print(f"P-value: {results.pvalues['gradur_master_exp']:.4f}")

# Save regression results
df_results = pd.DataFrame({
   'coefficient': results.params,
   'std_err': results.bse,
   'p_value': results.pvalues,
   't_stat': results.tvalues
})
print("\nFull regression results saved in df_results")

In [ ]:
df_final['exp'].unique()

In [ ]:
import matplotlib.pyplot as plt
# Check distribution of experience in regression sample
exp_counts = df_final['exp'].value_counts().sort_index()
print("Number of observations for each experience year:")
print(exp_counts)

# More detailed breakdown by degree type
exp_degree_counts = df_final.groupby(['exp', 'DGRDG'])['log_salary'].count().unstack()
exp_degree_counts.columns = ['Bachelor\'s', 'Master\'s']
print("\nBreakdown by degree type:")
print(exp_degree_counts)

# Calculate percentages
exp_percentages = (exp_counts / len(df_final) * 100).round(2)
print("\nPercentage of observations for each experience year:")
print(exp_percentages)

# Optional: Visualize the distribution
plt.figure(figsize=(10, 6))
exp_degree_counts.plot(kind='bar')
plt.title('Number of Observations by Experience Year and Degree Type')
plt.xlabel('Years of Experience')
plt.ylabel('Number of Observations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Create diagnostic variables
diagnostic_df = pd.DataFrame()

# Create experience dummies
exp_dummies = pd.get_dummies(df_final['exp'], prefix='exp')

# Create unemployment-experience interactions
ur_exp_interactions = pd.DataFrame()
for exp in df_final['exp'].unique():
    ur_exp_interactions[f'ur_exp_{exp}'] = df_final['gradur'] * (df_final['exp'] == exp)

# Combine for correlation analysis
diagnostic_vars = pd.concat([exp_dummies, ur_exp_interactions], axis=1)

# Calculate correlation matrix
corr_matrix = diagnostic_vars.corr()

# Print high correlations
print("High correlations (>0.7):")
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i,j]) > 0.7:
            print(f"{corr_matrix.columns[i]} & {corr_matrix.columns[j]}: {corr_matrix.iloc[i,j]:.3f}")

# Calculate VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

vif_results = calculate_vif(diagnostic_vars)
print("\nVariance Inflation Factors:")
print(vif_results.sort_values('VIF', ascending=False).head())

In [ ]:
df_final[['REFYR','BAYR']]

In [ ]:
df_final

In [ ]:
# Create unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in df_final['exp'].unique():
   ur_exp_interactions[f'ur_exp_{exp}'] = df_final['gradur'] * (df_final['exp'] == exp)

# Create master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in df_final['exp'].unique():
   master_exp_interactions[f'master_exp_{exp}'] = df_final['master'] * (df_final['exp'] == exp)

# Create master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in df_final['exp'].unique():
   master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_final['gradur'] * df_final['master'] * (df_final['exp'] == exp)

# Create year dummies
year_dummies = pd.get_dummies(df_final['REFYR'], prefix='year')
base_year = min(df_final['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables for regression
X = pd.concat([
   ur_exp_interactions,  # experience-specific unemployment effects
   master_exp_interactions,  # experience-specific master's premium
   master_ur_exp_interactions,  # experience-specific master's-unemployment effects
   year_dummies  # year fixed effects
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_final['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print(results.summary())

# Print effects by experience year
print("\nEffects by experience year:")
for exp in sorted(df_final['exp'].unique()):
   print(f"\nExperience {exp} years:")
   
   # Bachelor's unemployment effect
   ur_coef = f'ur_exp_{exp}'
   if ur_coef in results.params:
       print(f"Bachelor's unemployment effect:")
       print(f"Coefficient: {results.params[ur_coef]:.4f}")
       print(f"Standard Error: {results.bse[ur_coef]:.4f}")
       print(f"P-value: {results.pvalues[ur_coef]:.4f}")
   
   # Master's premium
   master_coef = f'master_exp_{exp}'
   if master_coef in results.params:
       print(f"Master's premium:")
       print(f"Coefficient: {results.params[master_coef]:.4f}")
       print(f"Standard Error: {results.bse[master_coef]:.4f}")
       print(f"P-value: {results.pvalues[master_coef]:.4f}")
   
   # Additional unemployment effect for master's
   master_ur_coef = f'master_ur_exp_{exp}'
   if master_ur_coef in results.params:
       print(f"Additional master's unemployment effect:")
       print(f"Coefficient: {results.params[master_ur_coef]:.4f}")
       print(f"Standard Error: {results.bse[master_ur_coef]:.4f}")
       print(f"P-value: {results.pvalues[master_ur_coef]:.4f}")

# restrict sample size to experience year [3-15]


In [ ]:
# First show sample sizes for potential ranges
print("Sample sizes by experience year:")
for exp in sorted(df_final['exp'].unique()):
   bach_count = len(df_final[(df_final['exp'] == exp) & (df_final['DGRDG'] == 1)])
   mast_count = len(df_final[(df_final['exp'] == exp) & (df_final['DGRDG'] == 2)])
   total = bach_count + mast_count
   print(f"Experience {exp}: Total = {total} (Bachelor's = {bach_count}, Master's = {mast_count})")

# Run regression for main sample (3-15 years)
df_main = df_final[df_final['exp'].between(3, 15)]

# Create variables for regression
# Unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   ur_exp_interactions[f'ur_exp_{exp}'] = df_main['gradur'] * (df_main['exp'] == exp)

# Master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_exp_interactions[f'master_exp_{exp}'] = df_main['master'] * (df_main['exp'] == exp)

# Master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_main['gradur'] * df_main['master'] * (df_main['exp'] == exp)

# Year fixed effects
year_dummies = pd.get_dummies(df_main['REFYR'], prefix='year')
base_year = min(df_main['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Run main regression (3-15 years)
X_main = pd.concat([
   ur_exp_interactions,
   master_exp_interactions,
   master_ur_exp_interactions,
   year_dummies
], axis=1)

X_main = sm.add_constant(X_main)
model_main = sm.OLS(df_main['log_salary'], X_main)
results_main = model_main.fit(cov_type='HC1')

print("\nMain Results (Experience 3-15):")
print(results_main.summary())

In [ ]:
# Main sample (3-15 years) with gender and race controls
df_main = df_final[df_final['exp'].between(3, 15)]

# Create gender dummy (assuming F is the reference category)
df_main['male'] = (df_main['GENDER'] == 'M').astype(int)

# Create race dummies (let's say reference category is white)
race_dummies = pd.get_dummies(df_main['RACEM'], prefix='race')
race_dummies = race_dummies.drop('race_4', axis=1)  # drop white as reference

# Create other variables as before
# Unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   ur_exp_interactions[f'ur_exp_{exp}'] = df_main['gradur'] * (df_main['exp'] == exp)

# Master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_exp_interactions[f'master_exp_{exp}'] = df_main['master'] * (df_main['exp'] == exp)

# Master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_main['gradur'] * df_main['master'] * (df_main['exp'] == exp)

# Year fixed effects
year_dummies = pd.get_dummies(df_main['REFYR'], prefix='year')
base_year = min(df_main['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables including new controls
X_main = pd.concat([
   ur_exp_interactions,
   master_exp_interactions,
   master_ur_exp_interactions,
   race_dummies,
   df_main[['male']],
   year_dummies
], axis=1)

X_main = sm.add_constant(X_main)

# Run regression
model_main = sm.OLS(df_main['log_salary'], X_main)
results_main = model_main.fit(cov_type='HC1')

print("\nResults with Gender and Race Controls (Experience 3-15):")
print(results_main.summary())

# Print gender and race effects
print("\nGender and Race Effects:")
for var in ['male'] + [col for col in X_main.columns if col.startswith('race_')]:
   if var in results_main.params:
       print(f"\n{var}:")
       print(f"Coefficient: {results_main.params[var]:.4f}")
       print(f"Standard Error: {results_main.bse[var]:.4f}")
       print(f"P-value: {results_main.pvalues[var]:.4f}")

In [ ]:
# First create race_dummies before checking correlations
df_main = df_final[df_final['exp'].between(3, 15)]
df_main['male'] = (df_main['GENDER'] == 'M').astype(int)
race_dummies = pd.get_dummies(df_main['RACEM'], prefix='race')
race_dummies = race_dummies.drop('race_4', axis=1)  # drop white as reference

# Now check correlations
print("Correlation between master's degree and gender/race:")
correlation_data = pd.concat([df_main[['master', 'male']], race_dummies], axis=1)
print(correlation_data.corr()['master'])

# Check distribution of master's degrees by gender and race
print("\nMaster's degree distribution by gender:")
print(pd.crosstab(df_main['master'], df_main['GENDER'], normalize='columns'))

print("\nMaster's degree distribution by race:")
print(pd.crosstab(df_main['master'], df_main['RACEM'], normalize='columns'))

# Additional check: Sample sizes by gender and race
print("\nSample sizes:")
print("By gender:")
print(df_main.groupby(['GENDER', 'master']).size())
print("\nBy race:")
print(df_main.groupby(['RACEM', 'master']).size())

In [ ]:
# Look at salary distribution in our main sample (3-15 years experience)
df_main = df_final[df_final['exp'].between(3, 15)]

print("Salary Distribution Summary:")
print(df_main['SALARY'].describe())

# Method 1: Using percentiles (e.g., 1st and 99th)
p1 = df_main['SALARY'].quantile(0.01)
p99 = df_main['SALARY'].quantile(0.99)
print(f"\n1st percentile: ${p1:,.2f}")
print(f"99th percentile: ${p99:,.2f}")

# Method 2: Using standard deviations (e.g., ±3 SD from mean)
mean = df_main['SALARY'].mean()
std = df_main['SALARY'].std()
lower_bound_sd = mean - 3*std
upper_bound_sd = mean + 3*std
print(f"\nMean ± 3SD bounds:")
print(f"Lower bound: ${lower_bound_sd:,.2f}")
print(f"Upper bound: ${upper_bound_sd:,.2f}")

# Let's see how many observations we would keep with each method
n_original = len(df_main)
n_percentile = len(df_main[(df_main['SALARY'] >= p1) & (df_main['SALARY'] <= p99)])
n_sd = len(df_main[(df_main['SALARY'] >= lower_bound_sd) & (df_main['SALARY'] <= upper_bound_sd)])

print(f"\nSample sizes:")
print(f"Original: {n_original:,}")
print(f"After percentile trimming: {n_percentile:,} ({n_percentile/n_original*100:.1f}%)")
print(f"After SD trimming: {n_sd:,} ({n_sd/n_original*100:.1f}%)")

In [ ]:
# Remove salary outliers for each survey year
df_main = df_final[df_final['exp'].between(3, 15)].copy()

# Function to get valid salary range for each year
def get_valid_salary_range(data, year):
   year_data = data[data['REFYR'] == year]
   p1 = year_data['SALARY'].quantile(0.01)
   p99 = year_data['SALARY'].quantile(0.99)
   return p1, p99

# Print sample sizes before trimming
print("Before trimming:")
print(df_main.groupby('REFYR').size())

# Remove outliers by year
clean_dfs = []
for year in df_main['REFYR'].unique():
   year_data = df_main[df_main['REFYR'] == year].copy()
   p1, p99 = get_valid_salary_range(df_main, year)
   print(f"\nYear {year}:")
   print(f"1st percentile: ${p1:,.2f}")
   print(f"99th percentile: ${p99:,.2f}")
   clean_year_data = year_data[(year_data['SALARY'] >= p1) & (year_data['SALARY'] <= p99)]
   clean_dfs.append(clean_year_data)

# Combine cleaned data
df_clean = pd.concat(clean_dfs)

print("\nAfter trimming:")
print(df_clean.groupby('REFYR').size())
print(f"\nTotal rows removed: {len(df_main) - len(df_clean)}")
print(f"Percentage of data retained: {(len(df_clean)/len(df_main))*100:.1f}%")

# Now run the regression with cleaned data
# [Previous regression code with df_clean instead of df_main]

## ***remove outliers (salary outside 1%-99% each REFYR) 

In [ ]:
# Run regression with cleaned data (experience 3-15 years)

# Create variables for regression
# Unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   ur_exp_interactions[f'ur_exp_{exp}'] = df_clean['gradur'] * (df_clean['exp'] == exp)

# Master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_exp_interactions[f'master_exp_{exp}'] = df_clean['master'] * (df_clean['exp'] == exp)

# Master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_clean['gradur'] * df_clean['master'] * (df_clean['exp'] == exp)

# Year fixed effects
year_dummies = pd.get_dummies(df_clean['REFYR'], prefix='year')
base_year = min(df_clean['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables
X = pd.concat([
   ur_exp_interactions,
   master_exp_interactions,
   master_ur_exp_interactions,
   year_dummies
], axis=1)

X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_clean['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print(results.summary())

# Print interpretation for key coefficients
print("\nEffects by experience year:")
for exp in range(3, 16):
   print(f"\nExperience {exp} years:")
   
   # Bachelor's unemployment effect
   ur_coef = f'ur_exp_{exp}'
   if ur_coef in results.params:
       print(f"Bachelor's unemployment effect:")
       print(f"Coefficient: {results.params[ur_coef]:.4f}")
       print(f"Standard Error: {results.bse[ur_coef]:.4f}")
       print(f"P-value: {results.pvalues[ur_coef]:.4f}")
   
   # Master's premium
   master_coef = f'master_exp_{exp}'
   if master_coef in results.params:
       print(f"Master's premium:")
       print(f"Coefficient: {results.params[master_coef]:.4f}")
       print(f"Standard Error: {results.bse[master_coef]:.4f}")
       print(f"P-value: {results.pvalues[master_coef]:.4f}")
   
   # Additional unemployment effect for master's
   master_ur_coef = f'master_ur_exp_{exp}'
   if master_ur_coef in results.params:
       print(f"Additional master's unemployment effect:")
       print(f"Coefficient: {results.params[master_ur_coef]:.4f}")
       print(f"Standard Error: {results.bse[master_ur_coef]:.4f}")
       print(f"P-value: {results.pvalues[master_ur_coef]:.4f}")

## + experience FE, + cohort trend (multicollinearity, less significant!)

In [ ]:
# Create experience dummies (experience fixed effects)
exp_dummies = pd.get_dummies(df_clean['exp'], prefix='exp')
base_exp = min(df_clean['exp'])
exp_dummies = exp_dummies.drop(f'exp_{base_exp}', axis=1)

# Create cohort trends
df_clean['cohort_trend'] = df_clean['BAYR'] - min(df_clean['BAYR'])  # linear trend
df_clean['cohort_trend2'] = df_clean['cohort_trend']**2  # quadratic trend

# Create unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
    ur_exp_interactions[f'ur_exp_{exp}'] = df_clean['gradur'] * (df_clean['exp'] == exp)

# Master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
    master_exp_interactions[f'master_exp_{exp}'] = df_clean['master'] * (df_clean['exp'] == exp)

# Master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
    master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_clean['gradur'] * df_clean['master'] * (df_clean['exp'] == exp)

# Year fixed effects
year_dummies = pd.get_dummies(df_clean['REFYR'], prefix='year')
base_year = min(df_clean['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables
X = pd.concat([
    ur_exp_interactions,
    master_exp_interactions,
    master_ur_exp_interactions,
    exp_dummies,  # experience fixed effects
    df_clean[['cohort_trend', 'cohort_trend2']],  # cohort trends
    year_dummies
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_clean['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print(results.summary())

# Print interpretation for key coefficients
print("\nExperience Fixed Effects:")
for col in X.columns:
    if col.startswith('exp_'):
        print(f"{col}: {results.params[col]:.4f} (p-value: {results.pvalues[col]:.4f})")

print("\nCohort Trends:")
print(f"Linear trend: {results.params['cohort_trend']:.4f} (p-value: {results.pvalues['cohort_trend']:.4f})")
print(f"Quadratic trend: {results.params['cohort_trend2']:.4f} (p-value: {results.pvalues['cohort_trend2']:.4f})")

In [ ]:
df_clean['CTZN']

# Subsample: white, male, citizen

In [ ]:
# Create subsample of white, male, citizens
df_sub = df_clean[
   (df_clean['RACEM'] == 4) &      # white
   (df_clean['GENDER'] == 'M') &   # male  
   (df_clean['CTZN'].isin([1, 2]))] # citizen

# Check sample size
print("Original sample size:", len(df_clean))
print("Subsample size:", len(df_sub))
print("\nPercentage kept:", f"{(len(df_sub)/len(df_clean))*100:.1f}%")

# Run regression on subsample
# Create variables for regression
# Unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   ur_exp_interactions[f'ur_exp_{exp}'] = df_sub['gradur'] * (df_sub['exp'] == exp)

# Master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_exp_interactions[f'master_exp_{exp}'] = df_sub['master'] * (df_sub['exp'] == exp)

# Master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_sub['gradur'] * df_sub['master'] * (df_sub['exp'] == exp)

# Year fixed effects
year_dummies = pd.get_dummies(df_sub['REFYR'], prefix='year')
base_year = min(df_sub['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables
X = pd.concat([
   ur_exp_interactions,
   master_exp_interactions,
   master_ur_exp_interactions,
   year_dummies
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_sub['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print("\nRegression Results:")
print(results.summary())

In [ ]:
# Create subsample of white, male, citizens
df_sub = df_clean[
#    (df_clean['RACEM'] == 1) ]     # white
   (df_clean['GENDER'] == 'M')]   # male  
#    (df_clean['CTZN'].isin([1,2])) # citizen

# Check sample size
print("Original sample size:", len(df_clean))
print("Subsample size:", len(df_sub))
print("\nPercentage kept:", f"{(len(df_sub)/len(df_clean))*100:.1f}%")

# Run regression on subsample
# Create variables for regression
# Unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   ur_exp_interactions[f'ur_exp_{exp}'] = df_sub['gradur'] * (df_sub['exp'] == exp)

# Master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_exp_interactions[f'master_exp_{exp}'] = df_sub['master'] * (df_sub['exp'] == exp)

# Master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
   master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_sub['gradur'] * df_sub['master'] * (df_sub['exp'] == exp)

# Year fixed effects
year_dummies = pd.get_dummies(df_sub['REFYR'], prefix='year')
base_year = min(df_sub['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables
X = pd.concat([
   ur_exp_interactions,
   master_exp_interactions,
   master_ur_exp_interactions,
   year_dummies
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_sub['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print("\nRegression Results:")
print(results.summary())

## + field-of-study FEs

In [ ]:
df_sub = df_clean
# Check unique values in NBAMEMG
print("Available fields in our subsample:")
print(df_sub['NBAMEMG'].value_counts().sort_index())

# Create field dummies without explicitly dropping any field
field_dummies = pd.get_dummies(df_sub['NBAMEMG'], prefix='field')

# Print column names of field dummies
print("\nField dummy columns:")
print(field_dummies.columns)

# Then drop the last field as reference (whatever it is)
last_field = field_dummies.columns[-1]
field_dummies = field_dummies.drop(last_field, axis=1)

# Continue with the rest of the regression code...

In [ ]:
# Create field dummies and drop field_7.0 as reference
field_dummies = pd.get_dummies(df_sub['NBAMEMG'], prefix='field')
field_dummies = field_dummies.drop('field_7.0', axis=1)

# Create other variables
# Unemployment rate interactions with experience
ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
    ur_exp_interactions[f'ur_exp_{exp}'] = df_sub['gradur'] * (df_sub['exp'] == exp)

# Master's degree interactions with experience
master_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
    master_exp_interactions[f'master_exp_{exp}'] = df_sub['master'] * (df_sub['exp'] == exp)

# Master's-unemployment-experience interactions
master_ur_exp_interactions = pd.DataFrame()
for exp in range(3, 16):
    master_ur_exp_interactions[f'master_ur_exp_{exp}'] = df_sub['gradur'] * df_sub['master'] * (df_sub['exp'] == exp)

# Year fixed effects
year_dummies = pd.get_dummies(df_sub['REFYR'], prefix='year')
base_year = min(df_sub['REFYR'])
year_dummies = year_dummies.drop(f'year_{base_year}', axis=1)

# Combine all variables including field FE
X = pd.concat([
    ur_exp_interactions,
    master_exp_interactions,
    master_ur_exp_interactions,
    field_dummies,  # field fixed effects (non-S&E as reference)
    year_dummies
], axis=1)

# Add constant
X = sm.add_constant(X)

# Run regression
model = sm.OLS(df_sub['log_salary'], X)
results = model.fit(cov_type='HC1')

# Print results
print(results.summary())

# Print field effects (compared to non-S&E fields)
print("\nField Fixed Effects (relative to non-S&E fields):")
for col in X.columns:
    if col.startswith('field_'):
        print(f"{col}: {results.params[col]:.4f} (p-value: {results.pvalues[col]:.4f})")

# visualization

In [ ]:
get_salary_effects(2)

In [ ]:
plt.figure(figsize=(12, 6))

# Use lighter blue
for i, ur in enumerate([2, 4]):  # two scenarios
   ba_effects, ma_effects = get_salary_effects(ur)
   
   color = '#1f77b4' if ur==2 else 'orange'  # lighter blue
   
   # Bachelor's: solid line
   plt.plot(x, ba_effects, color=color, linestyle='-', marker='o', 
            markersize=4, linewidth=1.5,
            label=f"Bachelor's (UR+{ur}pp)")
   
   # Master's: dashed line
   plt.plot(x, ma_effects, color=color, linestyle='--', marker='s', 
            markersize=4, linewidth=1.5,
            label=f"Master's (UR+{ur}pp)")

plt.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
plt.title('Salary Effects Under Different Unemployment Scenarios', pad=15)
plt.xlabel('Years of Experience')
plt.ylabel('Log Salary Effect')

plt.legend(loc='center right', frameon=True, framealpha=0.9)
plt.grid(True, alpha=0.2)

plt.ylim(min(min(ba_effects), min(ma_effects)) - 0.1, 
        max(max(ma_effects), max(ba_effects)) + 0.1)

plt.tight_layout()
plt.show()

In [ ]:
df_sub = df_clean
# Create graduation year dummies
grad_year_dummies = pd.get_dummies(df_sub['BAYR'], prefix='grad_year')

# Run regression of log salary on graduation year dummies
resid_model = sm.OLS(df_sub['log_salary'], grad_year_dummies)
resid_results = resid_model.fit()

# Get residuals
df_sub['earnings_residuals'] = resid_results.resid

# Calculate average residuals by graduation year and experience
avg_resid = df_sub.groupby(['BAYR', 'exp'])['earnings_residuals'].mean().reset_index()

# Plot
plt.figure(figsize=(12, 6))

# Left y-axis for earnings residuals
ax1 = plt.gca()
ax2 = ax1.twinx()

# Plot residuals for different experience years
experience_years = [3, 4, 5] + list(range(6, 11))  # 3,4,5 separately, 6-10 combined
colors = ['red', 'orange', 'yellow', 'blue']

for i, exp in enumerate([3, 4, 5]):
    exp_data = avg_resid[avg_resid['exp'] == exp]
    ax1.plot(exp_data['BAYR'], exp_data['earnings_residuals'], 
             marker='o', linestyle='-', label=f'Exp = {exp}', 
             color=colors[i])

# Combine 6-10 years experience
exp_610_data = avg_resid[avg_resid['exp'].between(6, 10)].groupby('BAYR')['earnings_residuals'].mean()
ax1.plot(exp_610_data.index, exp_610_data.values, 
         marker='o', linestyle='-', label='Exp = 6-10', 
         color=colors[3])

# Plot unemployment rate on right y-axis
ax2.plot(df_sub['BAYR'].unique(), df_sub.groupby('BAYR')['gradur'].first(), 
         color='navy', linewidth=2, label='UR at graduation')

# Labels and legend
ax1.set_xlabel('Year of graduation')
ax1.set_ylabel('Earnings residuals')
ax2.set_ylabel('Unemployment Rate (%)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Earnings by Experience Year for Different Graduation Cohorts')
plt.show()

In [ ]:
df_sub['earnings_residuals']

In [ ]:
# Same code for residuals calculation...
# Create plot with centered y-axes
plt.figure(figsize=(12, 6))

# Left y-axis for earnings residuals
ax1 = plt.gca()
ax2 = ax1.twinx()

# Plot residuals for different experience years
experience_years = [3, 4, 5] + list(range(6, 11))
colors = ['red', 'orange', 'yellow', 'blue']

for i, exp in enumerate([3, 4, 5]):
    exp_data = avg_resid[avg_resid['exp'] == exp]
    ax1.plot(exp_data['BAYR'], exp_data['earnings_residuals'], 
             marker='o', linestyle='-', label=f'Exp = {exp}', 
             color=colors[i])

# Combine 6-10 years experience
exp_610_data = avg_resid[avg_resid['exp'].between(6, 10)].groupby('BAYR')['earnings_residuals'].mean()
ax1.plot(exp_610_data.index, exp_610_data.values, 
         marker='o', linestyle='-', label='Exp = 6-10', 
         color=colors[3])

# Plot unemployment rate on right y-axis
# Center unemployment rate by subtracting mean
ur_data = df_sub.groupby('BAYR')['gradur'].first()
ur_centered = ur_data - ur_data.mean()
ax2.plot(ur_data.index, ur_centered, 
         color='navy', linewidth=2, label='UR at graduation')

# Add horizontal line at 0 for both axes
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Set limits to be symmetric around 0 for both axes
max_abs_resid = max(abs(avg_resid['earnings_residuals'].max()), 
                   abs(avg_resid['earnings_residuals'].min()))
max_abs_ur = max(abs(ur_centered.max()), abs(ur_centered.min()))

ax1.set_ylim(-max_abs_resid*1.1, max_abs_resid*1.1)
ax2.set_ylim(-max_abs_ur*1.1, max_abs_ur*1.1)

# Labels and legend
ax1.set_xlabel('Year of graduation')
ax1.set_ylabel('Earnings residuals')
ax2.set_ylabel('Unemployment Rate (deviation from mean)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Earnings by Experience Year for Different Graduation Cohorts')
plt.show()

In [ ]:
df_sub['earnings_residuals']

In [ ]:
# Same code for residuals calculation...
# Include more controls when calculating residuals
X_resid = pd.concat([
    grad_year_dummies,  # graduation year FE
    field_dummies,      # field of study FE
    year_dummies       # survey year FE
], axis=1)

# Calculate residuals
resid_model = sm.OLS(df_sub['log_salary'], X_resid)
resid_results = resid_model.fit()
df_sub['earnings_residuals'] = resid_results.resid
# Create plot with centered y-axes
plt.figure(figsize=(12, 6))

# Left y-axis for earnings residuals
ax1 = plt.gca()
ax2 = ax1.twinx()

# Plot residuals for different experience years
experience_years = [3, 4, 5] + list(range(6, 11))
colors = ['red', 'orange', 'yellow', 'blue']

for i, exp in enumerate([3, 4, 5]):
    exp_data = avg_resid[avg_resid['exp'] == exp]
    ax1.plot(exp_data['BAYR'], exp_data['earnings_residuals'], 
             marker='o', linestyle='-', label=f'Exp = {exp}', 
             color=colors[i])

# Combine 6-10 years experience
exp_610_data = avg_resid[avg_resid['exp'].between(6, 10)].groupby('BAYR')['earnings_residuals'].mean()
ax1.plot(exp_610_data.index, exp_610_data.values, 
         marker='o', linestyle='-', label='Exp = 6-10', 
         color=colors[3])

# Plot unemployment rate on right y-axis
# Center unemployment rate by subtracting mean
ur_data = df_sub.groupby('BAYR')['gradur'].first()
ur_centered = ur_data - ur_data.mean()
ax2.plot(ur_data.index, ur_centered, 
         color='navy', linewidth=2, label='UR at graduation')

# Add horizontal line at 0 for both axes
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Set limits to be symmetric around 0 for both axes
max_abs_resid = max(abs(avg_resid['earnings_residuals'].max()), 
                   abs(avg_resid['earnings_residuals'].min()))
max_abs_ur = max(abs(ur_centered.max()), abs(ur_centered.min()))

ax1.set_ylim(-max_abs_resid*1.1, max_abs_resid*1.1)
ax2.set_ylim(-max_abs_ur*1.1, max_abs_ur*1.1)

# Labels and legend
ax1.set_xlabel('Year of graduation')
ax1.set_ylabel('Earnings residuals')
ax2.set_ylabel('Unemployment Rate (deviation from mean)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Earnings by Experience Year for Different Graduation Cohorts')
plt.show()

In [ ]:
len(df_clean)

In [ ]:
# First calculate residuals from regressing log earnings on graduation year fixed effects
# Only for bachelor's degree holders
df_bach = df_clean[df_clean['DGRDG'] == 1]

# Create graduation year dummies
grad_year_dummies = pd.get_dummies(df_bach['BAYR'], prefix='grad_year')

# Calculate residuals
resid_model = sm.OLS(df_bach['log_salary'], grad_year_dummies)
resid_results = resid_model.fit()
df_bach['earnings_residuals'] = resid_results.resid

# Create experience groups
df_bach['exp_group'] = pd.cut(df_bach['exp'], 
                           bins=[2, 5, 8, 12, 15], 
                           labels=['3-5', '6-8', '9-12', '13-15'])

# Calculate average residuals by graduation year and experience group
avg_resid = df_bach.groupby(['BAYR', 'exp_group'])['earnings_residuals'].mean().reset_index()

# Calculate unemployment rate deviation from mean
ur_data = df_bach.groupby('BAYR')['gradur'].first()
ur_centered = ur_data - ur_data.mean()

# Create plot
plt.figure(figsize=(12, 6))
ax1 = plt.gca()
ax2 = ax1.twinx()

# Plot residuals for different experience groups
colors = ['red', 'orange', 'green', 'blue']
for i, exp_group in enumerate(avg_resid['exp_group'].unique()):
   group_data = avg_resid[avg_resid['exp_group'] == exp_group]
   ax1.plot(group_data['BAYR'], group_data['earnings_residuals'], 
            marker='o', linestyle='-', label=f'Exp = {exp_group}', 
            color=colors[i])

# Plot centered unemployment rate
ax2.plot(ur_data.index, ur_centered, 
        color='navy', linewidth=2, linestyle='-', 
        label='UR at graduation')

# Add horizontal line at 0
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Set axis labels and title
ax1.set_xlabel('Year of graduation')
ax1.set_ylabel('Earnings residuals')
ax2.set_ylabel('Unemployment Rate (deviation from mean)')

# Adjust y-axis limits to be symmetric around 0
max_abs_resid = max(abs(avg_resid['earnings_residuals'].max()), 
                   abs(avg_resid['earnings_residuals'].min()))
max_abs_ur = max(abs(ur_centered.max()), abs(ur_centered.min()))

ax1.set_ylim(-max_abs_resid*1.1, max_abs_resid*1.1)
ax2.set_ylim(-max_abs_ur*1.1, max_abs_ur*1.1)

# Add legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Earnings by Experience Year for Different Graduation Cohorts')
plt.show()

In [ ]:
# First calculate residuals from regressing log earnings on graduation year fixed effects
# Only for bachelor's degree holders
df_bach = df_clean[df_clean['DGRDG'] == 1]

# Create graduation year dummies
grad_year_dummies = pd.get_dummies(df_bach['BAYR'], prefix='grad_year')

# Calculate residuals
resid_model = sm.OLS(df_bach['log_salary'], grad_year_dummies)
resid_results = resid_model.fit()
df_bach['earnings_residuals'] = resid_results.resid

# Create experience groups
df_bach['exp_group'] = pd.cut(df_bach['exp'], 
                            bins=[2, 5, 8, 12, 15], 
                            labels=['3-5', '6-8', '9-12', '13-15'])

# Calculate average residuals by graduation year and experience group
avg_resid = df_bach.groupby(['BAYR', 'exp_group'])['earnings_residuals'].mean().reset_index()

# Calculate year-over-year change in unemployment rate
ur_data = df_bach.groupby('BAYR')['gradur'].first()
ur_change = ur_data - ur_data.shift(1)

# Create plot
plt.figure(figsize=(12, 6))
ax1 = plt.gca()
ax2 = ax1.twinx()

# Plot residuals for different experience groups
colors = ['red', 'orange', 'green', 'blue']
for i, exp_group in enumerate(avg_resid['exp_group'].unique()):
    group_data = avg_resid[avg_resid['exp_group'] == exp_group]
    ax1.plot(group_data['BAYR'], group_data['earnings_residuals'], 
             marker='o', linestyle='-', label=f'Exp = {exp_group}', 
             color=colors[i])

# Plot unemployment rate change
ax2.plot(ur_change.index, ur_change.values, 
         color='navy', linewidth=2, linestyle='-', 
         label='Change in UR')

# Add horizontal line at 0
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Set axis labels and title
ax1.set_xlabel('Year of graduation')
ax1.set_ylabel('Earnings residuals')
ax2.set_ylabel('Change in Unemployment Rate (pp)')

# Adjust y-axis limits to be symmetric around 0
max_abs_resid = max(abs(avg_resid['earnings_residuals'].max()), 
                    abs(avg_resid['earnings_residuals'].min()))
max_abs_ur = max(abs(ur_change.max()), abs(ur_change.min()))

ax1.set_ylim(-max_abs_resid*1.1, max_abs_resid*1.1)
ax2.set_ylim(-max_abs_ur*1.1, max_abs_ur*1.1)

# Add legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Earnings by Experience Year and Change in Unemployment Rate at Graduation')
plt.show()

# Print the unemployment rate changes
print("\nYear-over-year changes in unemployment rate:")
print(ur_change.sort_index())

In [ ]:
# Let's compare both approaches
df_bach = df_clean[df_clean['DGRDG'] == 1]
# Create experience groups
df_bach['exp_group'] = pd.cut(df_bach['exp'], 
                            bins=[2, 5, 8, 12, 15], 
                            labels=['3-5', '6-8', '9-12', '13-15'])

# Approach 1: Simple Average Log Salary
avg_salary = df_bach.groupby(['BAYR', 'exp_group'])['log_salary'].mean().reset_index()

# Approach 2: Residuals from regression on grad year dummies
grad_year_dummies = pd.get_dummies(df_bach['BAYR'], prefix='grad_year')
resid_model = sm.OLS(df_bach['log_salary'], grad_year_dummies)
resid_results = resid_model.fit()
df_bach['earnings_residuals'] = resid_results.resid
avg_resid = df_bach.groupby(['BAYR', 'exp_group'])['earnings_residuals'].mean().reset_index()

# Calculate unemployment rate change
ur_data = df_bach.groupby('BAYR')['gradur'].first()
ur_change = ur_data - ur_data.shift(1)

# Create two plots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))

# Plot 1: Average Log Salary
colors = ['red', 'orange', 'green', 'blue']
ax1_twin = ax1.twinx()

for i, exp_group in enumerate(avg_salary['exp_group'].unique()):
   group_data = avg_salary[avg_salary['exp_group'] == exp_group]
   ax1.plot(group_data['BAYR'], group_data['log_salary'], 
            marker='o', linestyle='-', label=f'Exp = {exp_group}', 
            color=colors[i])

ax1_twin.plot(ur_change.index, ur_change.values, 
             color='navy', linewidth=2, linestyle='-', 
             label='Change in UR')

ax1.set_xlabel('Year of graduation')
ax1.set_ylabel('Average Log Salary')
ax1_twin.set_ylabel('Change in Unemployment Rate (pp)')
ax1.set_title('Average Log Salary by Experience Group')
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# Plot 2: Earnings Residuals
ax2_twin = ax2.twinx()

for i, exp_group in enumerate(avg_resid['exp_group'].unique()):
   group_data = avg_resid[avg_resid['exp_group'] == exp_group]
   ax2.plot(group_data['BAYR'], group_data['earnings_residuals'], 
            marker='o', linestyle='-', label=f'Exp = {exp_group}', 
            color=colors[i])

ax2_twin.plot(ur_change.index, ur_change.values, 
             color='navy', linewidth=2, linestyle='-', 
             label='Change in UR')

ax2.set_xlabel('Year of graduation')
ax2.set_ylabel('Earnings Residuals')
ax2_twin.set_ylabel('Change in Unemployment Rate (pp)')
ax2.set_title('Earnings Residuals by Experience Group')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

# Add horizontal line at 0 for residuals plot
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nSummary Statistics for Both Approaches:")
print("\nAverage Log Salary range by experience group:")
for exp_group in avg_salary['exp_group'].unique():
   group_data = avg_salary[avg_salary['exp_group'] == exp_group]
   print(f"\nExp group {exp_group}:")
   print(f"Min: {group_data['log_salary'].min():.3f}")
   print(f"Max: {group_data['log_salary'].max():.3f}")
   print(f"Range: {group_data['log_salary'].max() - group_data['log_salary'].min():.3f}")

print("\nEarnings Residuals range by experience group:")
for exp_group in avg_resid['exp_group'].unique():
   group_data = avg_resid[avg_resid['exp_group'] == exp_group]
   print(f"\nExp group {exp_group}:")
   print(f"Min: {group_data['earnings_residuals'].min():.3f}")
   print(f"Max: {group_data['earnings_residuals'].max():.3f}")
   print(f"Range: {group_data['earnings_residuals'].max() - group_data['earnings_residuals'].min():.3f}")